# 🏆 2026 FIFA World Cup Predictor
Monte Carlo simulation using ELO-based match probabilities.
Runs 50,000 full tournament simulations to estimate each team's win probability.

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

N_SIMULATIONS = 50_000

# ELO ratings (FIFA ranking-based estimates, March 2026)
# Unknown playoff qualifiers assigned median ELO of 1650
ELO = {
    'Mexico': 1890, 'South Africa': 1600, 'South Korea': 1755, 'UEFA Playoff D': 1650,
    'Canada': 1750, 'Switzerland': 1850, 'Qatar': 1500, 'UEFA Playoff A': 1650,
    'Brazil': 2100, 'Morocco': 1820, 'Scotland': 1705, 'Haiti': 1400,
    'United States': 1880, 'Paraguay': 1680, 'Australia': 1720, 'UEFA Playoff C': 1650,
    'Germany': 2040, "Cote d'Ivoire": 1760, 'Ecuador': 1780, 'Curacao': 1550,
    'Netherlands': 1970, 'Japan': 1860, 'Tunisia': 1670, 'UEFA Playoff B': 1650,
    'Belgium': 1885, 'Egypt': 1700, 'Iran': 1650, 'New Zealand': 1580,
    'Spain': 2000, 'Uruguay': 1870, 'Saudi Arabia': 1700, 'Cape Verde': 1520,
    'France': 2045, 'Senegal': 1830, 'Norway': 1715, 'Interconf Playoff 2': 1650,
    'Argentina': 2060, 'Austria': 1800, 'Algeria': 1655, 'Jordan': 1480,
    'Portugal': 1975, 'Colombia': 1790, 'Uzbekistan': 1505, 'Interconf Playoff 1': 1650,
    'England': 2005, 'Croatia': 1855, 'Ghana': 1600, 'Panama': 1530,
}

# 2026 World Cup group assignments (12 groups of 4)
GROUPS = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'UEFA Playoff D'],
    'B': ['Canada', 'Switzerland', 'Qatar', 'UEFA Playoff A'],
    'C': ['Brazil', 'Morocco', 'Scotland', 'Haiti'],
    'D': ['United States', 'Paraguay', 'Australia', 'UEFA Playoff C'],
    'E': ['Germany', "Cote d'Ivoire", 'Ecuador', 'Curacao'],
    'F': ['Netherlands', 'Japan', 'Tunisia', 'UEFA Playoff B'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Uruguay', 'Saudi Arabia', 'Cape Verde'],
    'I': ['France', 'Senegal', 'Norway', 'Interconf Playoff 2'],
    'J': ['Argentina', 'Austria', 'Algeria', 'Jordan'],
    'K': ['Portugal', 'Colombia', 'Uzbekistan', 'Interconf Playoff 1'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

ALL_TEAMS = [t for teams in GROUPS.values() for t in teams]
print(f'Setup complete: {len(ALL_TEAMS)} teams across {len(GROUPS)} groups')

In [ ]:
# Load pre-calculated group stage probabilities from CSV
df = pd.read_csv('future_match_probabilities_baseline.csv')

# Normalize team names to match ELO dict
NAME_MAP = {
    "Cote d'Ivoire": "Cote d'Ivoire",
    "Côte d'Ivoire": "Cote d'Ivoire",
    'Curaçao': 'Curacao',
    'Cape_Verde': 'Cape Verde',
    'UEFA_Playoff_A': 'UEFA Playoff A',
    'UEFA_Playoff_B': 'UEFA Playoff B',
    'UEFA_Playoff_C': 'UEFA Playoff C',
    'UEFA_Playoff_D': 'UEFA Playoff D',
    'Interconf_Playoff_1': 'Interconf Playoff 1',
    'Interconf_Playoff_2': 'Interconf Playoff 2',
}

df['home_team'] = df['home_team'].map(lambda x: NAME_MAP.get(x, x))
df['away_team'] = df['away_team'].map(lambda x: NAME_MAP.get(x, x))

# Build bidirectional probability lookup: (team1, team2) -> (p1_win, p_draw, p2_win)
GROUP_PROBS = {}
for _, row in df.iterrows():
    h, a = row['home_team'], row['away_team']
    GROUP_PROBS[(h, a)] = (row['p_home_win'], row['p_draw'], row['p_away_win'])
    GROUP_PROBS[(a, h)] = (row['p_away_win'], row['p_draw'], row['p_home_win'])

print(f'Loaded {len(df)} group stage matchup probabilities')
print(f'Groups: {sorted(df["group"].unique())}')

In [ ]:
# ─── Core Simulation Functions ────────────────────────────────────────────────

def elo_ko_prob(team1, team2):
    """P(team1 wins) in knockout match (no draws) using ELO ratings."""
    e1 = ELO.get(team1, 1650)
    e2 = ELO.get(team2, 1650)
    return 1.0 / (1.0 + 10.0 ** ((e2 - e1) / 400.0))


def simulate_group_match(t1, t2, rng):
    """Simulate one group stage match.
    Returns (pts1, pts2, gd1, gd2) - points and goal difference.
    """
    if (t1, t2) in GROUP_PROBS:
        p1, pd_, p2 = GROUP_PROBS[(t1, t2)]
    else:
        # Fallback: ELO-based estimate
        diff = ELO.get(t1, 1650) - ELO.get(t2, 1650)
        base = 1.0 / (1.0 + 10.0 ** (-diff / 400.0))
        pd_ = max(0.10, 0.28 * (1 - abs(diff) / 1200))
        p1 = base * (1 - pd_) / (base + (1 - base))  # scale to fit
        p1 = max(0.05, min(0.90, (base - 0.5) * 0.9 + 0.5 - pd_ / 2))
        p2 = max(0.05, 1 - p1 - pd_)
        total = p1 + pd_ + p2
        p1, pd_, p2 = p1 / total, pd_ / total, p2 / total

    r = rng.random()
    margin = int(rng.integers(1, 4))  # 1–3 goal margin for wins
    if r < p1:
        return 3, 0, margin, -margin
    elif r < p1 + pd_:
        return 1, 1, 0, 0
    else:
        return 0, 3, -margin, margin


def simulate_group(teams, rng):
    """Simulate a full group stage. Returns team standings (sorted by pts, GD)."""
    pts = defaultdict(int)
    gd  = defaultdict(int)
    gf  = defaultdict(int)

    for i, t1 in enumerate(teams):
        for t2 in teams[i + 1:]:
            p1, p2, gd1, gd2 = simulate_group_match(t1, t2, rng)
            pts[t1] += p1; pts[t2] += p2
            gd[t1]  += gd1; gd[t2]  += gd2
            gf[t1]  += max(0, gd1);  gf[t2]  += max(0, gd2)

    # Sort: points DESC, goal difference DESC, goals for DESC, then random (fair coin)
    tiebreak = {t: rng.random() for t in teams}
    standings = sorted(teams,
                       key=lambda t: (pts[t], gd[t], gf[t], tiebreak[t]),
                       reverse=True)
    return standings, {t: (pts[t], gd[t], gf[t]) for t in teams}


def simulate_ko(team1, team2, rng):
    """Simulate one knockout match. Returns winner."""
    p = elo_ko_prob(team1, team2)
    return team1 if rng.random() < p else team2


def simulate_tournament(rng):
    """Run one full 2026 FIFA World Cup simulation. Returns the champion."""

    # ── Group Stage ───────────────────────────────────────────────────────────
    group_results = {}  # group_letter -> [1st, 2nd, 3rd, 4th]
    third_place_teams = []  # (pts, gd, gf, team) for 3rd place comparison

    for g, teams in GROUPS.items():
        standings, stats = simulate_group(teams, rng)
        group_results[g] = standings
        # Record 3rd place team's stats for best-3rd selection
        t3 = standings[2]
        third_place_teams.append((stats[t3][0], stats[t3][1], stats[t3][2], t3, g))

    # Best 8 of 12 third-place finishers advance
    third_place_teams.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    best_thirds = {row[3]: row[4] for row in third_place_teams[:8]}  # team -> source_group

    # ── Build positional lookup: position code -> actual team ─────────────────
    pos = {}  # e.g. '1A' -> 'Brazil', '2A' -> 'Morocco'
    for g, standings in group_results.items():
        pos[f'1{g}'] = standings[0]
        pos[f'2{g}'] = standings[1]

    # Assign 3rd place teams to bracket slots
    # Official 2026 bracket has 8 "3rd place" slots labeled by eligible source groups.
    # We assign each qualifying 3rd place team to its correct slot based on source group.
    THIRD_SLOTS = {
        '3ABCDF': ['A', 'B', 'C', 'D', 'F'],
        '3CDFGH': ['C', 'D', 'F', 'G', 'H'],
        '3CEFHI': ['C', 'E', 'F', 'H', 'I'],
        '3EHIJK': ['E', 'H', 'I', 'J', 'K'],
        '3AEHIJ': ['A', 'E', 'H', 'I', 'J'],
        '3BEFIJ': ['B', 'E', 'F', 'I', 'J'],
        '3EFGIJ': ['E', 'F', 'G', 'I', 'J'],
        '3DEIJL': ['D', 'E', 'I', 'J', 'L'],
    }

    # Match qualifying 3rd place teams to slots
    remaining_thirds = list(best_thirds.keys())
    rng.shuffle(remaining_thirds)  # shuffle to avoid assignment bias

    for slot_key, eligible_groups in THIRD_SLOTS.items():
        assigned = False
        for team in remaining_thirds:
            if best_thirds[team] in eligible_groups:
                pos[slot_key] = team
                remaining_thirds.remove(team)
                assigned = True
                break
        if not assigned:
            # Fallback: assign any remaining team
            if remaining_thirds:
                pos[slot_key] = remaining_thirds.pop(0)
            else:
                pos[slot_key] = 'TBD'

    def team(code):
        return pos.get(code, 'TBD')

    # ── Round of 32 ───────────────────────────────────────────────────────────
    r32 = [
        simulate_ko(team('2A'), team('2B'),       rng),
        simulate_ko(team('1C'), team('2F'),       rng),
        simulate_ko(team('1E'), team('3ABCDF'),   rng),
        simulate_ko(team('1F'), team('2C'),       rng),
        simulate_ko(team('2E'), team('2I'),       rng),
        simulate_ko(team('1I'), team('3CDFGH'),   rng),
        simulate_ko(team('1A'), team('3CEFHI'),   rng),
        simulate_ko(team('1L'), team('3EHIJK'),   rng),
        simulate_ko(team('1G'), team('3AEHIJ'),   rng),
        simulate_ko(team('1D'), team('3BEFIJ'),   rng),
        simulate_ko(team('1H'), team('2J'),       rng),
        simulate_ko(team('2K'), team('2L'),       rng),
        simulate_ko(team('1B'), team('3EFGIJ'),   rng),
        simulate_ko(team('2D'), team('2G'),       rng),
        simulate_ko(team('1J'), team('2H'),       rng),
        simulate_ko(team('1K'), team('3DEIJL'),   rng),
    ]

    # ── Round of 16 ───────────────────────────────────────────────────────────
    r16 = [
        simulate_ko(r32[0],  r32[3],  rng),   # 1
        simulate_ko(r32[2],  r32[5],  rng),   # 2
        simulate_ko(r32[1],  r32[4],  rng),   # 3
        simulate_ko(r32[6],  r32[7],  rng),   # 4
        simulate_ko(r32[11], r32[10], rng),   # 5
        simulate_ko(r32[9],  r32[8],  rng),   # 6
        simulate_ko(r32[14], r32[13], rng),   # 7
        simulate_ko(r32[12], r32[15], rng),   # 8
    ]

    # ── Quarter-Finals ────────────────────────────────────────────────────────
    qf = [
        simulate_ko(r16[0], r16[1], rng),
        simulate_ko(r16[2], r16[3], rng),
        simulate_ko(r16[4], r16[5], rng),
        simulate_ko(r16[6], r16[7], rng),
    ]

    # ── Semi-Finals ───────────────────────────────────────────────────────────
    sf = [
        simulate_ko(qf[0], qf[1], rng),
        simulate_ko(qf[2], qf[3], rng),
    ]

    # ── Final ─────────────────────────────────────────────────────────────────
    champion = simulate_ko(sf[0], sf[1], rng)
    return champion, sf, qf


print('Simulation functions ready.')

In [ ]:
# ─── Run Monte Carlo Simulation ───────────────────────────────────────────────
print(f'Running {N_SIMULATIONS:,} tournament simulations...')

rng = np.random.default_rng(seed=42)

champion_counts  = defaultdict(int)
final_counts     = defaultdict(int)
semifinal_counts = defaultdict(int)
quarterfinal_counts = defaultdict(int)

for i in range(N_SIMULATIONS):
    champion, sf, qf = simulate_tournament(rng)
    champion_counts[champion] += 1
    for t in sf:
        final_counts[t] += 1
    for t in qf:
        semifinal_counts[t] += 1

print(f'Done! Simulated {N_SIMULATIONS:,} World Cups.')
print(f'\nTop predicted champion: {max(champion_counts, key=champion_counts.get)}')

In [ ]:
# ─── Display Results ──────────────────────────────────────────────────────────
N = N_SIMULATIONS

results = []
for team in ALL_TEAMS:
    results.append({
        'Team': team,
        'ELO': ELO.get(team, 1650),
        'Win %':       round(champion_counts[team]  / N * 100, 2),
        'Final %':     round(final_counts[team]      / N * 100, 2),
        'Semi-Final %':round(semifinal_counts[team]  / N * 100, 2),
    })

results_df = pd.DataFrame(results).sort_values('Win %', ascending=False)
results_df = results_df[results_df['Win %'] > 0].reset_index(drop=True)
results_df.index = results_df.index + 1

print('=' * 60)
print('  2026 FIFA WORLD CUP PREDICTIONS')
print(f'  Based on {N:,} Monte Carlo simulations')
print('=' * 60)
display(results_df.to_string(index=True))

In [ ]:
# ─── Top 10 Contenders ────────────────────────────────────────────────────────
top10 = results_df.head(10).copy()

print('\n🏆 TOP 10 WORLD CUP CONTENDERS')
print('-' * 55)
print(f'{"#":<4} {"Team":<25} {"Win %":>7}  {"Final %":>8}  {"SF %":>7}')
print('-' * 55)
for rank, row in top10.iterrows():
    print(f'{rank:<4} {row["Team"]:<25} {row["Win %"]:>6.2f}%  {row["Final %"]:>7.2f}%  {row["Semi-Final %"]:>6.2f}%')
print('-' * 55)

winner = top10.iloc[0]
print(f'\n🥇 Predicted Winner: {winner["Team"]}  ({winner["Win %"]}% chance)')

In [1]:
# ─── Single Most-Likely Tournament Bracket ────────────────────────────────────
# Run one simulation with the top ELO team always winning (deterministic)

class DeterministicRNG:
    """Pseudo-RNG that always picks the higher-probability outcome."""
    def random(self): return 0.01    # always favors team1 (higher ELO)
    def integers(self, lo, hi): return (lo + hi) // 2
    def shuffle(self, x): pass       # no shuffling

det_rng = np.random.default_rng(seed=0)

# Deterministic bracket: always pick the team with higher ELO win prob
def simulate_group_det(teams, rng):
    """Deterministic group simulation - higher ELO teams always win."""
    pts = defaultdict(int)
    gd  = defaultdict(int)

    for i, t1 in enumerate(teams):
        for t2 in teams[i + 1:]:
            if (t1, t2) in GROUP_PROBS:
                p1, pd_, p2 = GROUP_PROBS[(t1, t2)]
            else:
                p1, pd_, p2 = (0.36, 0.28, 0.36)
            if p1 > p2:      pts[t1] += 3; gd[t1] += 1; gd[t2] -= 1
            elif p2 > p1:    pts[t2] += 3; gd[t2] += 1; gd[t1] -= 1
            else:            pts[t1] += 1; pts[t2] += 1

    standings = sorted(teams, key=lambda t: (pts[t], gd[t]), reverse=True)
    return standings, {t: (pts[t], gd[t], 0) for t in teams}


# Run deterministic simulation
pos_det = {}
third_det = []
for g, teams in GROUPS.items():
    standings, stats = simulate_group_det(teams, det_rng)
    pos_det[f'1{g}'] = standings[0]
    pos_det[f'2{g}'] = standings[1]
    t3 = standings[2]
    third_det.append((stats[t3][0], stats[t3][1], t3, g))

third_det.sort(reverse=True)
best_thirds_det = {row[2]: row[3] for row in third_det[:8]}

THIRD_SLOTS = {
    '3ABCDF': ['A','B','C','D','F'], '3CDFGH': ['C','D','F','G','H'],
    '3CEFHI': ['C','E','F','H','I'], '3EHIJK': ['E','H','I','J','K'],
    '3AEHIJ': ['A','E','H','I','J'], '3BEFIJ': ['B','E','F','I','J'],
    '3EFGIJ': ['E','F','G','I','J'], '3DEIJL': ['D','E','I','J','L'],
}
remaining = list(best_thirds_det.keys())
for slot_key, eligible in THIRD_SLOTS.items():
    for t in remaining:
        if best_thirds_det[t] in eligible:
            pos_det[slot_key] = t; remaining.remove(t); break
    else:
        if remaining: pos_det[slot_key] = remaining.pop(0)

def gt(code): return pos_det.get(code, 'TBD')
def ko(t1, t2):
    w = t1 if elo_ko_prob(t1, t2) >= 0.5 else t2
    return w

r32_d = [
    ko(gt('2A'), gt('2B')),    ko(gt('1C'), gt('2F')),
    ko(gt('1E'), gt('3ABCDF')),ko(gt('1F'), gt('2C')),
    ko(gt('2E'), gt('2I')),    ko(gt('1I'), gt('3CDFGH')),
    ko(gt('1A'), gt('3CEFHI')),ko(gt('1L'), gt('3EHIJK')),
    ko(gt('1G'), gt('3AEHIJ')),ko(gt('1D'), gt('3BEFIJ')),
    ko(gt('1H'), gt('2J')),    ko(gt('2K'), gt('2L')),
    ko(gt('1B'), gt('3EFGIJ')),ko(gt('2D'), gt('2G')),
    ko(gt('1J'), gt('2H')),    ko(gt('1K'), gt('3DEIJL')),
]
r16_d = [
    ko(r32_d[0],  r32_d[3]),   ko(r32_d[2],  r32_d[5]),
    ko(r32_d[1],  r32_d[4]),   ko(r32_d[6],  r32_d[7]),
    ko(r32_d[11], r32_d[10]),  ko(r32_d[9],  r32_d[8]),
    ko(r32_d[14], r32_d[13]),  ko(r32_d[12], r32_d[15]),
]
qf_d = [ko(r16_d[0],r16_d[1]), ko(r16_d[2],r16_d[3]),
        ko(r16_d[4],r16_d[5]), ko(r16_d[6],r16_d[7])]
sf_d = [ko(qf_d[0], qf_d[1]), ko(qf_d[2], qf_d[3])]
champ_d = ko(sf_d[0], sf_d[1])

print('\n🏆 MOST LIKELY BRACKET (deterministic - higher ELO always advances)')
print('=' * 50)
print(f'\n  SEMI-FINALS:')
print(f'    {qf_d[0]} vs {qf_d[1]}  →  {sf_d[0]}')
print(f'    {qf_d[2]} vs {qf_d[3]}  →  {sf_d[1]}')
print(f'\n  🥇 FINAL:  {sf_d[0]}  vs  {sf_d[1]}')
print(f'  🏆 WINNER: {champ_d}')
print(f'\n  Full QF line-up:')
for i, (t1, t2) in enumerate([(r16_d[0],r16_d[1]),(r16_d[2],r16_d[3]),
                                (r16_d[4],r16_d[5]),(r16_d[6],r16_d[7])], 1):
    print(f'    QF{i}: {t1:20} vs {t2:20}  →  {qf_d[i-1]}')

NameError: name 'np' is not defined